In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [3]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
from google.colab import files
uploaded=files.upload()

Saving shakespeare.txt to shakespeare.txt


In [5]:
with open("shakespeare.txt","r",encoding="utf-8") as f:
  text=f.read()

text[:500]

'\ufeff1603\n\nALLS WELL THAT ENDS WELL\n\nby William Shakespeare\n\nDramatis Personae\n\n  KING OF FRANCE\n  THE DUKE OF FLORENCE\n  BERTRAM, Count of Rousillon\n  LAFEU, an old lord\n  PAROLLES, a follower of Bertram\n  TWO FRENCH LORDS, serving with Bertram\n\n  STEWARD, Servant to the Countess of Rousillon\n  LAVACHE, a clown and Servant to the Countess of Rousillon\n  A PAGE, Servant to the Countess of Rousillon\n\n  COUNTESS OF ROUSILLON, mother to Bertram\n  HELENA, a gentlewoman protected by the Countess\n  A WIDO'

In [6]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '"', '$', '&', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '}', '\ufeff']
83


In [7]:
stoi={ch: i for i, ch in enumerate(chars)}
itos={i: ch for i, ch in enumerate(chars)}

In [8]:
print(stoi["a"])
print(itos[10])

55
-


In [9]:
data=torch.tensor([stoi[c] for c in text],dtype=torch.long)
print(data[:20])
print(data.shape)

tensor([82, 13, 18, 12, 15,  0,  0, 26, 37, 37, 44,  1, 48, 30, 37, 37,  1, 45,
        33, 26])
torch.Size([6347703])


In [10]:
block_size=8
x=data[:block_size]
y=data[1:block_size+1]

In [11]:
print(x)
print(y)

tensor([82, 13, 18, 12, 15,  0,  0, 26])
tensor([13, 18, 12, 15,  0,  0, 26, 37])


In [12]:
print("Input :", "".join(itos[i.item()] for i in x))
print("Target:", "".join(itos[i.item()] for i in y))

Input : ﻿1603

A
Target: 1603

AL


In [13]:
# 90% training, 10% validation
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print(train_data.shape)
print(val_data.shape)

torch.Size([5712932])
torch.Size([634771])


In [14]:
for t in range(block_size):
  context=x[:t+1]
  target=y[t]
  print(
        f"When input is '{''.join(itos[i.item()] for i in context)}'"
        f" --> predict '{itos[target.item()]}'"
    )

When input is '﻿' --> predict '1'
When input is '﻿1' --> predict '6'
When input is '﻿16' --> predict '0'
When input is '﻿160' --> predict '3'
When input is '﻿1603' --> predict '
'
When input is '﻿1603
' --> predict '
'
When input is '﻿1603

' --> predict 'A'
When input is '﻿1603

A' --> predict 'L'


In [15]:
batch_size=32
block_size=64

In [16]:
ix=torch.randint(
    len(data)-block_size,
    (batch_size,)
)

In [17]:
ix

tensor([5887083, 6017295,  132503, 5666636, 2457942, 1608025, 1680111, 2819821,
        4468064,  250894, 2741222, 1936295, 5895330, 6182641, 4859030, 5460163,
        3727352, 6117445, 2204562, 5304613, 5878224,  146652, 4194167, 1905834,
        5062848, 4917131, 5759250, 2666843, 5412328, 4802548, 4106085, 5872568])

In [18]:
x = torch.stack([
    data[i:i+block_size]
    for i in ix
])

y = torch.stack([
    data[i+1:i+block_size+1]
    for i in ix
])

In [19]:
y.shape

torch.Size([32, 64])

In [20]:
print("Input")
print("".join(itos[i.item()] for i in x[0]))

print()

print("Target")
print("".join(itos[i.item()] for i in y[0]))

Input
ten ere this day,
When I have heard your king's desert recounted

Target
en ere this day,
When I have heard your king's desert recounted,


In [21]:
def get_batch(data):
    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i+block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i+1:i+block_size+1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [22]:
xb, yb = get_batch(data)

print(xb.shape)
print(yb.shape)
xb.to(device)
yb.to(device)

torch.Size([32, 64])
torch.Size([32, 64])


tensor([[67,  1, 55,  ..., 74,  1, 77],
        [61,  1, 56,  ...,  1,  1, 40],
        [62, 59, 68,  ..., 63, 73, 62],
        ...,
        [30, 68, 61,  ..., 59, 58,  1],
        [58,  1, 60,  ..., 55, 74, 59],
        [59, 72, 59,  ..., 73,  1, 67]], device='cuda:0')

In [23]:
embedding=nn.Embedding(vocab_size,32).to(device)

In [24]:
embedding.weight.shape

torch.Size([83, 32])

In [25]:
emb = embedding(xb)

In [26]:
print(emb.shape)

torch.Size([32, 64, 32])


In [27]:
class Cl_RNN(nn.Module):
  def __init__(self,vocab_size,embedding_dim,hidden_size):
    super().__init__()

    self.embedding=nn.Embedding(
        vocab_size,
        embedding_dim
    )
    self.rnn=nn.RNN(
        input_size=embedding_dim,
        hidden_size=hidden_size,
        batch_first=True
    )
    self.fc = nn.Linear(
        hidden_size,
        vocab_size
    )

  def forward(self,x):
    x=self.embedding(x)
    output,hidden=self.rnn(x)
    logits=self.fc(output)
    return logits

In [28]:
embedding_dim=32
hidden_size=128
model=Cl_RNN(vocab_size,embedding_dim,hidden_size).to(device)

In [29]:
out=model(xb)
print(out.shape)

torch.Size([32, 64, 83])


In [30]:
output, hidden = model.rnn(model.embedding(xb))

print(output.shape)
print(hidden.shape)

torch.Size([32, 64, 128])
torch.Size([1, 32, 128])


In [31]:
out = model(xb)

print(out.shape)

torch.Size([32, 64, 83])


In [32]:
print(out[0,0])

tensor([ 0.0159,  0.0425,  0.0811,  0.1223, -0.0889,  0.1064, -0.1517, -0.0274,
        -0.1282, -0.2847, -0.0560,  0.0585,  0.0362, -0.0550, -0.1519,  0.0739,
        -0.0573, -0.1518, -0.1123,  0.0978, -0.1334, -0.0178, -0.1412, -0.0695,
         0.0062,  0.2202, -0.1551,  0.2653, -0.0554,  0.0340,  0.1166, -0.0950,
         0.2468,  0.1918, -0.0518,  0.0457,  0.1037,  0.0449, -0.0980, -0.1694,
        -0.1428, -0.1136, -0.2192,  0.1501, -0.1838,  0.0577, -0.1499, -0.0189,
         0.1761, -0.0644,  0.2192,  0.2096,  0.1143, -0.0964,  0.1759, -0.0053,
        -0.0090,  0.1785,  0.0491,  0.0054,  0.0289,  0.2113, -0.0610, -0.1612,
         0.0765,  0.1075,  0.0579, -0.2076, -0.0566,  0.2230,  0.1941,  0.0236,
        -0.0713, -0.1495, -0.1435,  0.0602, -0.0306, -0.0653,  0.2226,  0.2483,
         0.0593, -0.0628, -0.0128], device='cuda:0', grad_fn=<SelectBackward0>)


In [33]:
criterion = nn.CrossEntropyLoss()

In [34]:
print(out.shape)
print(yb.shape)

torch.Size([32, 64, 83])
torch.Size([32, 64])


In [35]:
logits = out.reshape(-1, vocab_size)

targets = yb.reshape(-1)

In [36]:
logits.shape

torch.Size([2048, 83])

In [37]:
targets.shape

torch.Size([2048])

In [38]:
loss = criterion(logits, targets)

print(loss)

tensor(4.4684, device='cuda:0', grad_fn=<NllLossBackward0>)


In [39]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [40]:
optimizer.zero_grad()

out = model(xb)

loss = criterion(
    out.reshape(-1, vocab_size),
    yb.reshape(-1)
)

loss.backward()

optimizer.step()

print(loss.item())

4.468419075012207


In [41]:
model.parameters()

<generator object Module.parameters at 0x7ddf8428cd60>

In [42]:
max_iters=5000

for step in range(max_iters):
  xb,yb=get_batch(train_data)
  logits=model(xb)
  loss=criterion(
      logits.reshape(-1,vocab_size),
      yb.reshape(-1)
  )
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if step % 100 == 0:
        print(step, loss.item())

0 4.4198713302612305
100 2.6386303901672363
200 2.270075798034668
300 2.1476526260375977
400 2.1730992794036865
500 2.009230852127075
600 1.971471905708313
700 1.9838221073150635
800 1.8753695487976074
900 1.7970714569091797
1000 1.8057782649993896
1100 1.8037627935409546
1200 1.8084886074066162
1300 1.7314033508300781
1400 1.8063926696777344
1500 1.7839871644973755
1600 1.8195180892944336
1700 1.6802871227264404
1800 1.630223274230957
1900 1.7838172912597656
2000 1.7155885696411133
2100 1.7281466722488403
2200 1.7861354351043701
2300 1.7196993827819824
2400 1.6730399131774902
2500 1.7207963466644287
2600 1.7609009742736816
2700 1.6321187019348145
2800 1.6315851211547852
2900 1.6670165061950684
3000 1.6478712558746338
3100 1.6772924661636353
3200 1.671006202697754
3300 1.6600027084350586
3400 1.687148094177246
3500 1.622204303741455
3600 1.6463277339935303
3700 1.705823540687561
3800 1.711430311203003
3900 1.7341434955596924
4000 1.5786155462265015
4100 1.635305643081665
4200 1.5919193

In [43]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    losses = {}

    for split in ["train", "val"]:

        data = train_data if split == "train" else val_data

        split_losses = []

        for _ in range(100):

            xb, yb = get_batch(data)

            logits = model(xb)

            loss = criterion(
                logits.reshape(-1, vocab_size),
                yb.reshape(-1)
            )

            split_losses.append(loss.item())

        losses[split] = sum(split_losses) / len(split_losses)

    model.train()

    return losses

In [44]:
max_iters = 50000

eval_interval = 500

for step in range(max_iters):

    if step % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {step:5d} | "
            f"Train Loss {losses['train']:.4f} | "
            f"Val Loss {losses['val']:.4f}"
        )

    xb, yb = get_batch(train_data)

    logits = model(xb)

    loss = criterion(
        logits.reshape(-1, vocab_size),
        yb.reshape(-1)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

Step     0 | Train Loss 1.6334 | Val Loss 1.8280
Step   500 | Train Loss 1.6130 | Val Loss 1.8339
Step  1000 | Train Loss 1.6120 | Val Loss 1.8120
Step  1500 | Train Loss 1.6039 | Val Loss 1.8190
Step  2000 | Train Loss 1.5963 | Val Loss 1.8016
Step  2500 | Train Loss 1.5832 | Val Loss 1.8072
Step  3000 | Train Loss 1.5890 | Val Loss 1.8039
Step  3500 | Train Loss 1.5843 | Val Loss 1.7971
Step  4000 | Train Loss 1.5794 | Val Loss 1.7853
Step  4500 | Train Loss 1.5782 | Val Loss 1.7908
Step  5000 | Train Loss 1.5648 | Val Loss 1.7782
Step  5500 | Train Loss 1.5647 | Val Loss 1.7852
Step  6000 | Train Loss 1.5639 | Val Loss 1.7843
Step  6500 | Train Loss 1.5553 | Val Loss 1.7892
Step  7000 | Train Loss 1.5564 | Val Loss 1.7771
Step  7500 | Train Loss 1.5618 | Val Loss 1.7717
Step  8000 | Train Loss 1.5538 | Val Loss 1.7676
Step  8500 | Train Loss 1.5539 | Val Loss 1.7651
Step  9000 | Train Loss 1.5442 | Val Loss 1.7571
Step  9500 | Train Loss 1.5372 | Val Loss 1.7651
Step 10000 | Train L

In [45]:
model.eval()

Cl_RNN(
  (embedding): Embedding(83, 32)
  (rnn): RNN(32, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=83, bias=True)
)

In [46]:
context = torch.zeros((1, 1), dtype=torch.long).to(device)

In [47]:
logits = model(context)

In [48]:
print(logits.shape)

torch.Size([1, 1, 83])


In [49]:
logits = logits[:, -1, :]

In [50]:
probs = F.softmax(logits, dim=-1)

In [51]:
print(probs.shape)

torch.Size([1, 83])


In [52]:
next_char = torch.multinomial(
    probs,
    num_samples=1
)

In [53]:
context = torch.cat(
    (context, next_char),
    dim=1
)

In [54]:
@torch.no_grad()
def generate(model, start_token, max_new_tokens):

    model.eval()

    context = torch.tensor([[start_token]], device=device)

    for _ in range(max_new_tokens):

        logits = model(context)

        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        context = torch.cat((context, next_token), dim=1)

    return context

In [55]:
generated = generate(
    model,
    start_token=0,
    max_new_tokens=500
)

In [56]:
text = "".join(
    itos[i.item()]
    for i in generated[0]
)

print(text)


When ever true, cause  
    And to foun a feedock.
    I have never heaven? Father of Land of say.                   Here
But
    My souls, prepoly, for fareweech

Emiceness by their man sound attended ment o' th' empssillest taught safore her hona's mean;
    And heil of gransming-
    But dram.
Lateal three are discount,
    Res son therefore Clarman. 'Tis the cark of seen; for't'no me of seeks back, none with yet yet in all
    an wars angelor; not suchal of him fell never stature destruct he


In [58]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}, "checkpoint.pth")

In [63]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = "/content/drive"

torch.save(model.state_dict(),
           SAVE_PATH + "checkpoint.pth")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pickle

with open(SAVE_PATH + "vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

with open(SAVE_PATH + "vocab.pkl", "rb") as f:
    vocab = pickle.load(f)